# RideBase V0 — Rule-Based Maintenance Baseline

Bu notebook RideBase Synthetic Dataset v1.2 üzerinde deterministik bakım politikaları kullanarak:

- sonraki servis zaman/km tahmini
- sonraki bakım task adayları

üretir.

**Sonuçlar yalnız sentetik offline PoC sonuçlarıdır. Production validation değildir.**

GitHub Issue #579 için amaç yüksek skor değil; sonraki V1/V2/V3 modellerinin aynı split üzerinde
geçmesi gereken açıklanabilir, leakage-free ve tekrarlanabilir bir referans oluşturmaktır. Bu notebook
ML/regression/survival modeli eğitmez, LLM veya rastgelelik kullanmaz.


## 1. Merkezi config, v1.2 version guard ve authoritative veri yükleme

`01_data_understanding.ipynb` ve `02_eda.ipynb` ile aynı merkezi path yaklaşımı kullanılır.
`ridebase-ml/data/` altındaki legacy kopyalar analiz kaynağı değildir. Version/count guard başarısızsa
notebook hemen durur.


In [1]:
from pathlib import Path
import gc
import hashlib
import json
import math
import os

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_rows", 160)
pd.set_option("display.max_colwidth", 180)
sns.set_theme(style="whitegrid", context="notebook")


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "notebooks").is_dir() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("ridebase-ml proje kökü bulunamadı")


PROJECT_ROOT = find_project_root()
DATASET_VERSION = "1.2.0"
RULE_VERSION = "V0_RULE_BASELINE_1.0.0"
DATASET_ROOT = PROJECT_ROOT.parent / "ridebase_v1_2"
SOURCE_DIR = DATASET_ROOT / "source_tables"
DERIVED_DIR = DATASET_ROOT / "derived_outputs"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORT_TABLES_DIR = REPORTS_DIR / "tables"
FIGURES_DIR = REPORTS_DIR / "figures" / "v0_rule_baseline"

for directory in [OUTPUTS_DIR, MODELS_DIR, REPORT_TABLES_DIR, FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

if not SOURCE_DIR.is_dir() or not DERIVED_DIR.is_dir():
    raise FileNotFoundError(f"Authoritative RideBase v1.2 release bulunamadı: {DATASET_ROOT}")

with open(DERIVED_DIR / "dataset_metadata.json", encoding="utf-8") as file:
    dataset_metadata = json.load(file)
with open(DERIVED_DIR / "quality_report.json", encoding="utf-8") as file:
    quality_report = json.load(file)

dataset_info = dataset_metadata.get("dataset", {})
dataset_version = dataset_info.get("dataset_version")
generator_version = dataset_info.get("generator_version")
quality_gate = quality_report.get("report", {}).get("release_gate")

if dataset_version != DATASET_VERSION:
    raise RuntimeError(f"Dataset version mismatch: {dataset_version!r} != {DATASET_VERSION!r}")
if generator_version != DATASET_VERSION:
    raise RuntimeError(f"Generator version mismatch: {generator_version!r} != {DATASET_VERSION!r}")
if quality_gate != "PASS":
    raise RuntimeError(f"Quality gate PASS değil: {quality_gate!r}")

motorcycles = pd.read_csv(SOURCE_DIR / "motorcycles.csv", encoding="utf-8-sig", low_memory=False)
motorcycle_models = pd.read_csv(SOURCE_DIR / "ridebase_motorcycle_models_v1.csv", encoding="utf-8-sig", low_memory=False)
usage_profiles = pd.read_csv(SOURCE_DIR / "usage_profiles.csv", encoding="utf-8-sig", low_memory=False)
maintenance_tasks = pd.read_csv(SOURCE_DIR / "maintenance_tasks.csv", encoding="utf-8-sig", low_memory=False)
maintenance_policies = pd.read_csv(SOURCE_DIR / "maintenance_policies.csv", encoding="utf-8-sig", low_memory=False)
services = pd.read_csv(SOURCE_DIR / "services.csv", encoding="utf-8-sig", low_memory=False)
services_enriched = pd.read_csv(SOURCE_DIR / "services_enriched.csv", encoding="utf-8-sig", low_memory=False)
service_tasks = pd.read_csv(SOURCE_DIR / "service_tasks.csv", encoding="utf-8-sig", low_memory=False)

ml_snapshots = pd.read_parquet(DERIVED_DIR / "ml_maintenance_snapshots.parquet", engine="pyarrow")
next_service_targets = pd.read_parquet(DERIVED_DIR / "ml_next_service_targets.parquet", engine="pyarrow")
next_task_targets = pd.read_parquet(DERIVED_DIR / "ml_next_task_targets.parquet", engine="pyarrow")
split_manifest = pd.read_csv(DERIVED_DIR / "split_manifest.csv", encoding="utf-8-sig", low_memory=False)

expected_counts = {"services": 41_518, "service_tasks": 204_000, "motorcycles": 10_000}
actual_counts = {
    "services": len(services), "service_tasks": len(service_tasks), "motorcycles": len(motorcycles)
}
for name, expected in expected_counts.items():
    if actual_counts[name] != expected:
        raise RuntimeError(f"v1.2 row-count guard failed: {name}={actual_counts[name]} != {expected}")
if len(services_enriched) != len(services) or set(services_enriched["service_id"]) != set(services["service_id"]):
    raise RuntimeError("services.csv ↔ services_enriched.csv key/row alignment failed")

version_validation = pd.DataFrame([
    {"check": "dataset_version", "actual": dataset_version, "expected": DATASET_VERSION, "status": "PASS"},
    {"check": "generator_version", "actual": generator_version, "expected": DATASET_VERSION, "status": "PASS"},
    {"check": "quality_gate", "actual": quality_gate, "expected": "PASS", "status": "PASS"},
] + [
    {"check": f"row_count::{name}", "actual": actual, "expected": expected_counts[name], "status": "PASS"}
    for name, actual in actual_counts.items()
])
display(version_validation)
print("Dataset Version Guard: PASS")
print("Authoritative root:", DATASET_ROOT)


,check,actual,expected,status
0,dataset_version,1.2.0,1.2.0,PASS
1,generator_version,1.2.0,1.2.0,PASS
2,quality_gate,PASS,PASS,PASS
3,row_count::services,41518,41518,PASS
4,row_count::service_tasks,204000,204000,PASS
5,row_count::motorcycles,10000,10000,PASS


Dataset Version Guard: PASS
Authoritative root: /Users/nihatkutukoglu/Downloads/ridebase_synthetic_dataset_v1/ridebase_v1_2


## 2. Gerçek schema keşfi ve policy contract

Aşağıdaki tablo kolon isimlerini doğrudan dosyalardan çıkarır. V0 policy engine gerçek v1.2 alanlarını
kullanır: model/group scope, canonical `task_code`, km/ay aralıkları, trigger mode, precedence ve
kullanım koşulu çarpanları.


In [2]:
loaded_frames = {
    "motorcycles.csv": motorcycles,
    "ridebase_motorcycle_models_v1.csv": motorcycle_models,
    "usage_profiles.csv": usage_profiles,
    "maintenance_tasks.csv": maintenance_tasks,
    "maintenance_policies.csv": maintenance_policies,
    "services.csv": services,
    "services_enriched.csv": services_enriched,
    "service_tasks.csv": service_tasks,
    "ml_maintenance_snapshots.parquet": ml_snapshots,
    "ml_next_service_targets.parquet": next_service_targets,
    "ml_next_task_targets.parquet": next_task_targets,
    "split_manifest.csv": split_manifest,
}
schema_inventory = pd.DataFrame([
    {
        "file": name,
        "rows": len(frame),
        "columns": frame.shape[1],
        "column_names": " | ".join(map(str, frame.columns)),
    }
    for name, frame in loaded_frames.items()
])
display(schema_inventory)

policy_semantics = pd.DataFrame([
    {"semantic": "task identifier", "actual_column": "task_code"},
    {"semantic": "model/group applicability", "actual_column": "scope_type + model_id + policy_group"},
    {"semantic": "kilometre interval", "actual_column": "initial_trigger_km + recurring_km + wear_mean_km"},
    {"semantic": "time interval", "actual_column": "initial_trigger_months + recurring_months"},
    {"semantic": "trigger combination", "actual_column": "trigger_mode"},
    {"semantic": "usage constraints", "actual_column": "severe_use_multiplier + courier_multiplier + offroad_multiplier + dusty_environment_multiplier"},
    {"semantic": "priority/type", "actual_column": "precedence + policy_kind + scope_type"},
])
display(policy_semantics)
display(maintenance_policies.groupby(["scope_type", "policy_kind", "trigger_mode"], dropna=False).size().reset_index(name="policy_count"))

required_columns = {
    "maintenance_policies": {"policy_id", "scope_type", "model_id", "policy_group", "task_code", "policy_kind", "recurring_km", "recurring_months", "trigger_mode", "precedence", "is_generator_active"},
    "service_tasks": {"service_id", "motorcycle_id", "task_code", "status", "completed", "completed_at", "task_role"},
    "services": {"service_id", "motorcycle_id", "odometer_km"},
    "snapshots": {"snapshot_id", "snapshot_at", "snapshot_odometer_km", "motorcycle_id", "model_id", "service_sequence"},
}
for frame_name, columns in required_columns.items():
    frame = {"maintenance_policies": maintenance_policies, "service_tasks": service_tasks, "services": services, "snapshots": ml_snapshots}[frame_name]
    missing = sorted(columns - set(frame.columns))
    if missing:
        raise RuntimeError(f"{frame_name} gerçek schema alanları eksik: {missing}")


,file,rows,columns,column_names
0,motorcycles.csv,10000,29,motorcycle_id | customer_id | workshop_id | model_id | brand | model_name | category | powertrain_type | production_year | production_year_basis | first_registration_date | own...
1,ridebase_motorcycle_models_v1.csv,39,38,model_id | brand | model_name | variant_or_generation | category | market_priority | generator_weight_raw | generator_weight_seed | weight_basis | powertrain_type | engine_disp...
2,usage_profiles.csv,10000,30,usage_profile_id | motorcycle_id | customer_id | workshop_id | usage_type | annual_km_baseline | annual_km_basis | city_ratio | highway_ratio | offroad_ratio | track_ratio | av...
3,maintenance_tasks.csv,98,17,task_code | canonical_name_tr | canonical_name_en | component_group | action_type | is_periodic | is_wear_based | is_fault_based | requires_part | applicable_powertrain | requi...
4,maintenance_policies.csv,621,30,policy_id | scope_type | model_id | policy_group | task_code | policy_kind | initial_trigger_km | recurring_km | initial_trigger_months | recurring_months | trigger_mode | seve...
5,services.csv,41518,43,service_id | workshop_id | customer_id | motorcycle_id | appointment_id | service_type_code | primary_trigger_task | trigger_due_km | trigger_due_date | service_delay_days | ar...
6,services_enriched.csv,41518,49,service_id | subtotal_before_discount | discount_rate | tax_rate | pricing_currency | cost_calculation_basis | cost_rule_version | workshop_id | customer_id | motorcycle_id | a...
7,service_tasks.csv,204000,27,service_task_id | service_id | motorcycle_id | workshop_id | task_sequence | task_code | display_title | title_variant_type | component_group | action_type | trigger_reason | i...
8,ml_maintenance_snapshots.parquet,41518,142,snapshot_id | source_service_id | snapshot_at | snapshot_date | service_sequence | snapshot_year | snapshot_month | snapshot_quarter | snapshot_day_of_year | snapshot_month_sin...
9,ml_next_service_targets.parquet,41518,38,snapshot_id | source_service_id | motorcycle_id | customer_id | workshop_id | snapshot_at | snapshot_odometer_km | target_event_observed | is_right_censored | next_service_id |...


,semantic,actual_column
0,task identifier,task_code
1,model/group applicability,scope_type + model_id + policy_group
2,kilometre interval,initial_trigger_km + recurring_km + wear_mean_km
3,time interval,initial_trigger_months + recurring_months
4,trigger combination,trigger_mode
5,usage constraints,severe_use_multiplier + courier_multiplier + offroad_multiplier + dusty_environment_multiplier
6,priority/type,precedence + policy_kind + scope_type


,scope_type,policy_kind,trigger_mode,policy_count
0,GROUP,CONDITION_BASED,CONDITION_ONLY,23
1,GROUP,SCHEDULED,KM_ONLY,33
2,GROUP,SCHEDULED,TIME_ONLY,20
3,GROUP,SCHEDULED,WHICHEVER_FIRST,379
4,GROUP,WEAR_BASED,CONDITION_ONLY,127
5,MODEL,SCHEDULED,KM_ONLY,13
6,MODEL,SCHEDULED,TIME_ONLY,3
7,MODEL,SCHEDULED,WHICHEVER_FIRST,23


## 3. Snapshot-safe input matrisi ve tamamlanmış task geçmişi

Prediction anı `snapshot_at`'tir. Task history yalnız `COMPLETED`, `completed=1` ve
`completed_at <= snapshot_at` olaylarından kurulur. `DECLINED` tasklar ve gelecekteki task/service
bilgileri dışarıda kalır. İlk bilinen baseline için immutable motorcycle tablosundaki
`observation_start_date + initial_mileage_km` kullanılır.


In [3]:
snapshot_columns = [
    "snapshot_id", "source_service_id", "snapshot_at", "motorcycle_id", "workshop_id",
    "snapshot_odometer_km", "service_sequence", "model_id", "brand", "model_name", "category",
    "powertrain_type", "final_drive_type", "cooling_type", "transmission_type", "policy_group",
    "motorcycle_age_years", "avg_km_per_day_since_previous_service", "previous_failure_count",
]
base_snapshots = ml_snapshots[snapshot_columns].copy()
base_snapshots["snapshot_at"] = pd.to_datetime(base_snapshots["snapshot_at"], errors="raise")
base_snapshots = base_snapshots.merge(
    motorcycles[["motorcycle_id", "first_registration_date", "observation_start_date", "initial_mileage_km"]],
    on="motorcycle_id", how="left", validate="many_to_one"
).merge(
    usage_profiles[["motorcycle_id", "usage_type", "annual_km_baseline", "riding_intensity", "offroad_ratio", "climate_zone", "profile_start_date"]],
    on="motorcycle_id", how="left", validate="many_to_one"
)
for column in ["first_registration_date", "observation_start_date", "profile_start_date"]:
    base_snapshots[column] = pd.to_datetime(base_snapshots[column], errors="coerce")

if base_snapshots["snapshot_id"].duplicated().any():
    raise RuntimeError("Snapshot key duplicate")
if (base_snapshots["profile_start_date"] > base_snapshots["snapshot_at"]).any():
    raise RuntimeError("Usage profile snapshot sonrasında başlıyor; leakage-safe baseline kurulamaz")

completed_tasks = service_tasks.loc[
    service_tasks["status"].eq("COMPLETED") & service_tasks["completed"].eq(1),
    ["service_id", "motorcycle_id", "task_code", "completed_at", "task_role"]
].copy()
completed_tasks = completed_tasks.merge(
    services[["service_id", "motorcycle_id", "odometer_km"]],
    on=["service_id", "motorcycle_id"], how="inner", validate="many_to_one"
)
completed_tasks["task_completed_at"] = pd.to_datetime(completed_tasks["completed_at"], errors="coerce")
completed_tasks = completed_tasks.dropna(subset=["task_completed_at", "odometer_km"])
completed_tasks = completed_tasks.sort_values(["motorcycle_id", "task_code", "task_completed_at", "service_id"])
completed_tasks["completion_count"] = completed_tasks.groupby(["motorcycle_id", "task_code"]).cumcount() + 1

declined_in_history = int(((service_tasks["status"] == "DECLINED") & service_tasks["service_id"].isin(completed_tasks["service_id"])).sum())
print("Base snapshots:", len(base_snapshots))
print("Completed task events:", len(completed_tasks))
print("DECLINED accepted as completed:", declined_in_history)


Base snapshots: 41518
Completed task events: 202585
DECLINED accepted as completed: 1415


## 4. Deterministic policy engine

Kurallar:

- Aynı task için `MODEL` policy, yüksek `precedence` ile `GROUP` policy'yi override eder.
- SCHEDULED policy'de ilk geçmiş yoksa `initial_trigger_*`, yoksa `recurring_*` kullanılır.
- WEAR_BASED policy için dağılımdan örnek çekilmez; deterministik `wear_mean_km` kullanılır.
- CONDITION_BASED ve hesaplanabilir interval içermeyen tasklar timing/task due adayı değildir.
- Kullanım çarpanları yalnız snapshot anında mevcut usage profile alanlarından deterministik uygulanır.
- Km→gün dönüşümünde önce geçmiş servis aralığının km/gün hızı, yoksa profile `annual_km_baseline / 365.25` kullanılır.
- Hem tarih hem km varsa `WHICHEVER_FIRST` uygulanır.
- Ana prediction `ACTIONABLE`: overdue horizon `0` yapılır. Negatif RAW horizon ayrıca saklanır.


In [4]:
MONTH_DAYS = 365.25 / 12
NEAR_DUE_DAYS = 30.0
NEAR_DUE_KM = 1_000.0
OFFROAD_RATIO_THRESHOLD = 0.10
DUSTY_CLIMATE_PATTERN = "ARID"
SAFETY_COMPONENTS = {"BRAKES", "TIRES_WHEELS", "STEERING", "SUSPENSION"}

policy_columns = [
    "policy_id", "scope_type", "model_id", "policy_group", "task_code", "policy_kind",
    "initial_trigger_km", "recurring_km", "initial_trigger_months", "recurring_months",
    "trigger_mode", "severe_use_multiplier", "courier_multiplier", "offroad_multiplier",
    "dusty_environment_multiplier", "wear_mean_km", "precedence", "confidence",
    "is_manufacturer_exact_interval", "is_generator_active",
]
task_columns = [
    "task_code", "canonical_name_tr", "component_group", "is_periodic", "is_wear_based",
    "is_fault_based", "applicable_powertrain", "required_final_drive", "required_cooling_type",
    "required_transmission_type", "can_be_next_service_target",
]
active_policies = maintenance_policies.loc[maintenance_policies["is_generator_active"].eq(1), policy_columns].merge(
    maintenance_tasks[task_columns], on="task_code", how="left", validate="many_to_one"
)


def applicable_policy_spec(model_id: str) -> pd.DataFrame:
    model = motorcycle_models.loc[motorcycle_models["model_id"].eq(model_id)]
    if model.empty:
        return pd.DataFrame(columns=active_policies.columns)
    model = model.iloc[0]
    group_rows = active_policies.loc[
        active_policies["scope_type"].eq("GROUP") & active_policies["policy_group"].eq(model["policy_group"])
    ]
    model_rows = active_policies.loc[
        active_policies["scope_type"].eq("MODEL") & active_policies["model_id"].eq(model_id)
    ]
    spec = pd.concat([group_rows, model_rows], ignore_index=True)
    if spec.empty:
        return spec
    spec = spec.sort_values(["task_code", "precedence", "policy_id"], ascending=[True, False, True]).drop_duplicates("task_code")
    mask = spec["applicable_powertrain"].isin(["BOTH", model["powertrain_type"]])
    for required, actual in [
        ("required_final_drive", "final_drive_type"),
        ("required_cooling_type", "cooling_type"),
        ("required_transmission_type", "transmission_type"),
    ]:
        mask &= spec[required].isna() | spec[required].eq(model[actual])
    return spec.loc[mask].reset_index(drop=True)


policy_specs_by_model = {model_id: applicable_policy_spec(model_id) for model_id in sorted(base_snapshots["model_id"].unique())}
model_policy_summary = pd.DataFrame([
    {
        "model_id": model_id,
        "category": motorcycle_models.set_index("model_id").loc[model_id, "category"],
        "applicable_policy_count": len(spec),
        "canonical_task_count": spec["task_code"].nunique(),
        "timing_capable_policy_count": int(
            spec[["initial_trigger_km", "recurring_km", "initial_trigger_months", "recurring_months", "wear_mean_km"]].notna().any(axis=1).sum()
        ),
    }
    for model_id, spec in policy_specs_by_model.items()
])
display(model_policy_summary)


def compute_model_predictions(model_id: str, model_snapshots: pd.DataFrame):
    spec = policy_specs_by_model[model_id]
    base_output_columns = [
        "snapshot_id", "motorcycle_id", "snapshot_at", "snapshot_odometer_km",
        "model_id", "model_name", "category", "workshop_id", "usage_type",
        "riding_intensity", "motorcycle_age_years", "service_sequence", "previous_failure_count",
    ]
    if spec.empty:
        unavailable = model_snapshots[base_output_columns].copy()
        unavailable["prediction_available"] = False
        unavailable["prediction_reason"] = "no applicable policy"
        unavailable["policy_count_considered"] = 0
        return unavailable, pd.DataFrame()

    spec_for_merge = spec.drop(columns=["model_id", "policy_group"], errors="ignore")
    left = model_snapshots.assign(_cross_key=1).merge(spec_for_merge.assign(_cross_key=1), on="_cross_key", how="inner").drop(columns="_cross_key")
    policy_count = len(spec)
    model_motorcycles = set(model_snapshots["motorcycle_id"])
    events = completed_tasks.loc[
        completed_tasks["motorcycle_id"].isin(model_motorcycles) & completed_tasks["task_code"].isin(set(spec["task_code"])),
        ["motorcycle_id", "task_code", "task_completed_at", "odometer_km", "completion_count", "task_role"]
    ].rename(columns={"odometer_km": "last_completed_task_odometer", "task_role": "last_task_role"})

    left = left.sort_values(["snapshot_at", "motorcycle_id", "task_code"])
    if events.empty:
        for column in ["task_completed_at", "last_completed_task_odometer", "completion_count", "last_task_role"]:
            left[column] = pd.NA
    else:
        events = events.sort_values(["task_completed_at", "motorcycle_id", "task_code"])
        left = pd.merge_asof(
            left,
            events,
            left_on="snapshot_at",
            right_on="task_completed_at",
            by=["motorcycle_id", "task_code"],
            direction="backward",
            allow_exact_matches=True,
        )

    has_history = left["completion_count"].fillna(0).gt(0)
    initial_km_interval = left["initial_trigger_km"].combine_first(left["recurring_km"]).combine_first(left["wear_mean_km"])
    recurring_km_interval = left["recurring_km"].combine_first(left["wear_mean_km"])
    initial_month_interval = left["initial_trigger_months"].combine_first(left["recurring_months"])
    recurring_month_interval = left["recurring_months"]
    left["base_interval_km"] = np.where(has_history, recurring_km_interval, initial_km_interval)
    left["base_interval_months"] = np.where(has_history, recurring_month_interval, initial_month_interval)

    usage_factor = np.ones(len(left), dtype=float)
    usage_factor *= np.where(left["riding_intensity"].eq("HIGH"), left["severe_use_multiplier"].fillna(1.0), 1.0)
    usage_factor *= np.where(left["usage_type"].eq("COURIER"), left["courier_multiplier"].fillna(1.0), 1.0)
    usage_factor *= np.where(left["offroad_ratio"].fillna(0).ge(OFFROAD_RATIO_THRESHOLD), left["offroad_multiplier"].fillna(1.0), 1.0)
    dusty = left["climate_zone"].astype("string").str.contains(DUSTY_CLIMATE_PATTERN, case=False, na=False)
    usage_factor *= np.where(dusty, left["dusty_environment_multiplier"].fillna(1.0), 1.0)
    left["usage_interval_factor"] = usage_factor

    deterministic_policy = left["policy_kind"].isin(["SCHEDULED", "WEAR_BASED"])
    left["interval_km"] = pd.to_numeric(left["base_interval_km"], errors="coerce") * usage_factor
    left["interval_days"] = pd.to_numeric(left["base_interval_months"], errors="coerce") * MONTH_DAYS * usage_factor
    left.loc[~deterministic_policy, ["interval_km", "interval_days"]] = np.nan

    left["baseline_task_odometer"] = pd.to_numeric(left["last_completed_task_odometer"], errors="coerce").where(has_history, left["initial_mileage_km"])
    left["baseline_task_date"] = pd.to_datetime(left["task_completed_at"], errors="coerce").where(has_history, left["observation_start_date"])
    left["next_due_km"] = left["baseline_task_odometer"] + left["interval_km"]
    left["next_due_date"] = left["baseline_task_date"] + pd.to_timedelta(left["interval_days"], unit="D")
    left["raw_remaining_km"] = left["next_due_km"] - left["snapshot_odometer_km"]
    left["raw_remaining_days_date"] = (left["next_due_date"] - left["snapshot_at"]).dt.total_seconds() / 86_400

    historical_speed = pd.to_numeric(left["avg_km_per_day_since_previous_service"], errors="coerce")
    profile_speed = pd.to_numeric(left["annual_km_baseline"], errors="coerce") / 365.25
    left["km_per_day_at_snapshot"] = historical_speed.where(historical_speed.gt(0), profile_speed.where(profile_speed.gt(0)))
    left["raw_remaining_days_from_km"] = left["raw_remaining_km"] / left["km_per_day_at_snapshot"]

    km_days = left["raw_remaining_days_from_km"]
    date_days = left["raw_remaining_days_date"]
    whichever = pd.concat([km_days, date_days], axis=1).min(axis=1, skipna=True)
    left["raw_due_horizon_days"] = np.select(
        [
            left["trigger_mode"].eq("KM_ONLY"),
            left["trigger_mode"].eq("TIME_ONLY"),
            left["trigger_mode"].eq("WHICHEVER_FIRST"),
            left["policy_kind"].eq("WEAR_BASED"),
        ],
        [km_days, date_days, whichever, whichever],
        default=np.nan,
    )
    left["trigger_type_candidate"] = np.where(
        km_days.notna() & date_days.notna(),
        np.where(km_days.le(date_days), "KM", "DATE"),
        np.where(km_days.notna(), "KM", np.where(date_days.notna(), "DATE", "UNAVAILABLE")),
    )

    future_count = int((pd.to_datetime(left["task_completed_at"], errors="coerce") > left["snapshot_at"]).sum())
    valid = left.loc[np.isfinite(pd.to_numeric(left["raw_due_horizon_days"], errors="coerce"))].copy()
    if valid.empty:
        unavailable = model_snapshots[base_output_columns].copy()
        unavailable["prediction_available"] = False
        unavailable["prediction_reason"] = "insufficient history or deterministic interval"
        unavailable["policy_count_considered"] = policy_count
        unavailable["future_history_leak_count"] = future_count
        return unavailable, pd.DataFrame()

    trigger = valid.sort_values(
        ["snapshot_id", "raw_due_horizon_days", "precedence", "task_code", "policy_id"],
        ascending=[True, True, False, True, True],
    ).drop_duplicates("snapshot_id").copy()
    trigger["raw_predicted_days_to_next_service"] = trigger["raw_due_horizon_days"]
    trigger["predicted_days_to_next_service"] = trigger["raw_due_horizon_days"].clip(lower=0)
    trigger["raw_predicted_km_to_next_service"] = trigger["raw_due_horizon_days"] * trigger["km_per_day_at_snapshot"]
    trigger["predicted_km_to_next_service"] = trigger["predicted_days_to_next_service"] * trigger["km_per_day_at_snapshot"]
    trigger["predicted_next_service_date"] = trigger["snapshot_at"] + pd.to_timedelta(trigger["predicted_days_to_next_service"], unit="D")
    trigger["predicted_next_service_odometer_km"] = trigger["snapshot_odometer_km"] + trigger["predicted_km_to_next_service"]
    trigger["is_overdue"] = trigger["raw_due_horizon_days"].lt(0)
    trigger["overdue_days"] = (-trigger["raw_due_horizon_days"]).clip(lower=0)
    trigger["overdue_km"] = (-trigger["raw_remaining_km"]).clip(lower=0).fillna(0)
    trigger["prediction_available"] = True
    trigger["prediction_reason"] = "available"
    trigger["policy_count_considered"] = policy_count
    trigger["future_history_leak_count"] = future_count
    trigger = trigger.rename(columns={
        "task_code": "trigger_task_code", "canonical_name_tr": "trigger_task_name",
        "trigger_type_candidate": "trigger_type", "policy_id": "trigger_policy_id",
    })

    prediction_columns = base_output_columns + [
        "prediction_available", "prediction_reason", "predicted_next_service_date",
        "predicted_next_service_odometer_km", "predicted_days_to_next_service",
        "predicted_km_to_next_service", "raw_predicted_days_to_next_service",
        "raw_predicted_km_to_next_service", "is_overdue", "overdue_days", "overdue_km",
        "trigger_task_code", "trigger_task_name", "trigger_type", "trigger_policy_id",
        "policy_count_considered", "km_per_day_at_snapshot", "future_history_leak_count",
    ]
    available = trigger[prediction_columns]
    missing = model_snapshots.loc[~model_snapshots["snapshot_id"].isin(set(available["snapshot_id"])), base_output_columns].copy()
    if not missing.empty:
        missing["prediction_available"] = False
        missing["prediction_reason"] = np.where(
            missing["snapshot_at"].isna(), "missing date",
            np.where(missing["snapshot_odometer_km"].isna(), "missing odometer", "insufficient history or deterministic interval"),
        )
        missing["policy_count_considered"] = policy_count
        missing["future_history_leak_count"] = future_count
    predictions = pd.concat([available, missing], ignore_index=True, sort=False)

    trigger_ref = available[["snapshot_id", "predicted_days_to_next_service", "predicted_km_to_next_service"]].rename(columns={
        "predicted_days_to_next_service": "service_horizon_days",
        "predicted_km_to_next_service": "service_horizon_km",
    })
    candidates = valid.merge(trigger_ref, on="snapshot_id", how="inner", validate="many_to_one")
    candidates["actionable_due_days"] = candidates["raw_due_horizon_days"].clip(lower=0)
    candidates["actionable_due_km"] = candidates["raw_remaining_km"].clip(lower=0)
    candidates["actionable_due_km"] = candidates["actionable_due_km"].fillna(candidates["actionable_due_days"] * candidates["km_per_day_at_snapshot"])
    candidates = candidates.loc[
        candidates["can_be_next_service_target"].eq(1)
        & (
            candidates["actionable_due_days"].le(candidates["service_horizon_days"] + NEAR_DUE_DAYS)
            | candidates["actionable_due_km"].le(candidates["service_horizon_km"] + NEAR_DUE_KM)
        )
    ].copy()
    candidates["is_overdue_task"] = candidates["raw_due_horizon_days"].lt(0)
    candidates["safety_priority"] = candidates["component_group"].isin(SAFETY_COMPONENTS).astype(int)
    candidates["rule_score"] = (
        candidates["is_overdue_task"].astype(float) * 1_000_000
        + candidates["actionable_due_days"].le(candidates["service_horizon_days"]).astype(float) * 100_000
        + candidates["safety_priority"] * 10_000
        + candidates["precedence"].fillna(0) * 10
        - candidates["actionable_due_days"].fillna(1e6)
        - candidates["actionable_due_km"].fillna(1e9) / 10_000
    )
    candidates = candidates.sort_values(
        ["snapshot_id", "rule_score", "task_code", "policy_id"],
        ascending=[True, False, True, True],
    )
    candidates["rule_rank"] = candidates.groupby("snapshot_id").cumcount() + 1
    candidates["reason"] = np.select(
        [
            candidates["is_overdue_task"],
            candidates["actionable_due_days"].le(candidates["service_horizon_days"]),
            candidates["actionable_due_km"].le(candidates["service_horizon_km"]),
        ],
        ["overdue", "due by predicted service date", "due by predicted service mileage"],
        default="near due window",
    )
    candidate_columns = [
        "snapshot_id", "motorcycle_id", "task_code", "canonical_name_tr", "component_group",
        "policy_id", "rule_score", "rule_rank", "reason", "raw_due_horizon_days",
        "raw_remaining_km", "actionable_due_days", "actionable_due_km", "next_due_date",
        "next_due_km", "is_overdue_task", "completion_count", "last_task_role",
    ]
    return predictions, candidates[candidate_columns]


prediction_chunks = []
candidate_chunks = []
for model_id, model_snapshot_rows in base_snapshots.groupby("model_id", sort=True):
    model_predictions, model_candidates = compute_model_predictions(model_id, model_snapshot_rows.copy())
    prediction_chunks.append(model_predictions)
    if not model_candidates.empty:
        candidate_chunks.append(model_candidates)

next_service_predictions = pd.concat(prediction_chunks, ignore_index=True, sort=False).sort_values("snapshot_id").reset_index(drop=True)
next_task_predictions_long = pd.concat(candidate_chunks, ignore_index=True, sort=False).sort_values(["snapshot_id", "rule_rank"]).reset_index(drop=True)
next_service_predictions["prediction_available"] = next_service_predictions["prediction_available"].fillna(False).astype(bool)
next_service_predictions["is_overdue"] = next_service_predictions["is_overdue"].fillna(False).astype(bool)
next_service_predictions["dataset_version"] = dataset_version
next_service_predictions["rule_version"] = RULE_VERSION
next_task_predictions_long["dataset_version"] = dataset_version
next_task_predictions_long["rule_version"] = RULE_VERSION

if len(next_service_predictions) != len(base_snapshots) or next_service_predictions["snapshot_id"].duplicated().any():
    raise RuntimeError("Next-service prediction snapshot coverage/key uniqueness failed")

display(next_service_predictions.head())
display(next_task_predictions_long.head(20))
print("Timing predictions:", len(next_service_predictions))
print("Timing coverage:", f"{next_service_predictions['prediction_available'].mean():.2%}")
print("Task candidate rows:", len(next_task_predictions_long))


,model_id,category,applicable_policy_count,canonical_task_count,timing_capable_policy_count
0,BAJAJ_DOMINAR250,TOURING,29,29,28
1,BAJAJ_NS125,NAKED,27,27,26
2,BAJAJ_NS200,NAKED,33,33,32
3,CFMOTO_250CLX,ROADSTER,29,29,28
4,CFMOTO_250DUAL,ADVENTURE,29,29,28
5,CFMOTO_250NK,NAKED,29,29,28
6,CFMOTO_250SR,SUPERSPORT,29,29,28
7,CFMOTO_450CLC,CRUISER,26,26,26
8,CFMOTO_450NK,NAKED,29,29,28
9,CFMOTO_650NK,NAKED,29,29,28


/var/folders/35/w7v4jvv930z1wthrlzsp49th0000gn/T/ipykernel_29684/3367863644.py:267: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  next_service_predictions["prediction_available"] = next_service_predictions["prediction_available"].fillna(False).astype(bool)
/var/folders/35/w7v4jvv930z1wthrlzsp49th0000gn/T/ipykernel_29684/3367863644.py:268: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  next_service_predictions["is_overdue"] = next_service_predictions["is_overdue"].fillna(False).astype(bool)


,snapshot_id,motorcycle_id,snapshot_at,snapshot_odometer_km,model_id,model_name,category,workshop_id,usage_type,riding_intensity,motorcycle_age_years,service_sequence,previous_failure_count,prediction_available,prediction_reason,predicted_next_service_date,predicted_next_service_odometer_km,predicted_days_to_next_service,predicted_km_to_next_service,raw_predicted_days_to_next_service,raw_predicted_km_to_next_service,is_overdue,overdue_days,overdue_km,trigger_task_code,trigger_task_name,trigger_type,trigger_policy_id,policy_count_considered,km_per_day_at_snapshot,future_history_leak_count,dataset_version,rule_version
0,SNP_SVC000001,MC000001,2024-07-04 16:08:00,84478,YAMAHA_NMAX125,NMAX 125,SCOOTER,WS0003,COMMUTER,HIGH,7.378508,1,0,True,available,2024-07-06 16:26:47.443669963,84580.0,2.013049,102.0,2.013049,102.0,False,0.000000,0.0,ENGINE_OIL_CHANGE,Motor Yağı Değişimi,KM,POL0071,30.0,50.669405,0.0,1.2.0,V0_RULE_BASELINE_1.0.0
1,SNP_SVC000002,MC000001,2025-08-11 17:12:00,105231,YAMAHA_NMAX125,NMAX 125,SCOOTER,WS0003,COMMUTER,HIGH,8.481862,2,0,True,available,2025-08-11 17:12:00.000000000,105231.0,0.000000,0.0,-329.205723,-16951.0,True,329.205723,16951.0,AIR_FILTER_INSPECTION,Hava Filtresi Kontrolü,KM,POL0072,30.0,51.490599,0.0,1.2.0,V0_RULE_BASELINE_1.0.0
2,SNP_SVC000003,MC000001,2026-03-26 09:36:00,117670,YAMAHA_NMAX125,NMAX 125,SCOOTER,WS0003,COMMUTER,HIGH,9.103354,3,0,True,available,2026-03-26 09:36:00.000000000,117670.0,0.000000,0.0,-535.591540,-29390.0,True,535.591540,29390.0,AIR_FILTER_INSPECTION,Hava Filtresi Kontrolü,KM,POL0072,30.0,54.873906,0.0,1.2.0,V0_RULE_BASELINE_1.0.0
3,SNP_SVC000004,MC000001,2026-06-24 15:01:00,122774,YAMAHA_NMAX125,NMAX 125,SCOOTER,WS0003,COMMUTER,HIGH,9.349760,4,0,True,available,2026-06-24 15:01:00.000000000,122774.0,0.000000,0.0,-583.249718,-32994.0,True,583.249718,32994.0,COOLANT_LEVEL_CHECK,Soğutma Sıvısı Seviye Kontrolü,KM,POL0093,30.0,56.569251,0.0,1.2.0,V0_RULE_BASELINE_1.0.0
4,SNP_SVC000005,MC000002,2023-09-19 16:01:00,30196,HONDA_PCX125,PCX125,SCOOTER,WS0008,TOURING,MEDIUM,2.442163,1,0,True,available,2023-09-19 16:01:00.000000000,30196.0,0.000000,0.0,-49.283771,-1756.0,True,49.283771,1756.0,ENGINE_OIL_CHANGE,Motor Yağı Değişimi,KM,POL0606,30.0,35.630390,0.0,1.2.0,V0_RULE_BASELINE_1.0.0


,snapshot_id,motorcycle_id,task_code,canonical_name_tr,component_group,policy_id,rule_score,rule_rank,reason,raw_due_horizon_days,raw_remaining_km,actionable_due_days,actionable_due_km,next_due_date,next_due_km,is_overdue_task,completion_count,last_task_role,dataset_version,rule_version
0,SNP_SVC000001,MC000001,ENGINE_OIL_CHANGE,Motor Yağı Değişimi,ENGINE,POL0071,1.000980e+05,1,due by predicted service date,2.013049,102.0,2.013049,102.0,2025-04-09 04:48:00,84580.0,False,NaN,NaN,1.2.0,V0_RULE_BASELINE_1.0.0
1,SNP_SVC000002,MC000001,BRAKE_FLUID_CHECK,Fren Hidroliği Kontrolü,BRAKES,POL0081,1.110100e+06,1,overdue,-300.074192,-15451.0,0.000000,0.0,2025-06-21 06:00:00,89780.0,True,NaN,NaN,1.2.0,V0_RULE_BASELINE_1.0.0
2,SNP_SVC000002,MC000001,FORK_INSPECTION,Ön Amortisör Kontrolü,SUSPENSION,POL0084,1.110100e+06,2,overdue,-300.074192,-15451.0,0.000000,0.0,2025-06-21 06:00:00,89780.0,True,NaN,NaN,1.2.0,V0_RULE_BASELINE_1.0.0
3,SNP_SVC000002,MC000001,FRONT_BRAKE_PAD_INSPECTION,Ön Fren Balatası Kontrolü,BRAKES,POL0079,1.110100e+06,3,overdue,-300.074192,-15451.0,0.000000,0.0,2025-06-21 06:00:00,89780.0,True,NaN,NaN,1.2.0,V0_RULE_BASELINE_1.0.0
4,SNP_SVC000002,MC000001,FRONT_TIRE_CHANGE,Ön Lastik Değişimi,TIRES_WHEELS,POL0090,1.110100e+06,4,overdue,-67.021943,-3451.0,0.000000,0.0,NaT,101780.0,True,NaN,NaN,1.2.0,V0_RULE_BASELINE_1.0.0
5,SNP_SVC000002,MC000001,FRONT_TIRE_INSPECTION,Ön Lastik Kontrolü,TIRES_WHEELS,POL0082,1.110100e+06,5,overdue,-300.074192,-15451.0,0.000000,0.0,2025-06-21 06:00:00,89780.0,True,NaN,NaN,1.2.0,V0_RULE_BASELINE_1.0.0
6,SNP_SVC000002,MC000001,REAR_BRAKE_PAD_INSPECTION,Arka Fren Balatası Kontrolü,BRAKES,POL0080,1.110100e+06,6,overdue,-300.074192,-15451.0,0.000000,0.0,2025-06-21 06:00:00,89780.0,True,NaN,NaN,1.2.0,V0_RULE_BASELINE_1.0.0
7,SNP_SVC000002,MC000001,REAR_TIRE_CHANGE,Arka Lastik Değişimi,TIRES_WHEELS,POL0091,1.110100e+06,7,overdue,-144.706026,-7451.0,0.000000,0.0,NaT,97780.0,True,NaN,NaN,1.2.0,V0_RULE_BASELINE_1.0.0
8,SNP_SVC000002,MC000001,REAR_TIRE_INSPECTION,Arka Lastik Kontrolü,TIRES_WHEELS,POL0083,1.110100e+06,8,overdue,-300.074192,-15451.0,0.000000,0.0,2025-06-21 06:00:00,89780.0,True,NaN,NaN,1.2.0,V0_RULE_BASELINE_1.0.0
9,SNP_SVC000002,MC000001,STEERING_BEARING_INSPECTION,Gidon Rulmanı Kontrolü,STEERING,POL0085,1.110100e+06,9,overdue,-300.074192,-15451.0,0.000000,0.0,2025-06-21 06:00:00,89780.0,True,NaN,NaN,1.2.0,V0_RULE_BASELINE_1.0.0


Timing predictions: 41518
Timing coverage: 100.00%
Task candidate rows: 748707


## 5. Next-task wide output ve prediction coverage

Candidate ranking tamamen deterministiktir: overdue → predicted service horizon içinde due → safety
component → policy precedence → yakınlık → alfabetik task code. Rastgele failure taskları tahmin edilmeye
çalışılmaz; bu, V0'ın bilinçli sınırıdır.


In [5]:
def pipe_join(values):
    return "|".join(values.astype(str).tolist())


task_aggregate = (
    next_task_predictions_long.groupby("snapshot_id", sort=False)
    .agg(
        predicted_task_codes=("task_code", pipe_join),
        predicted_task_count=("task_code", "size"),
    )
    .reset_index()
)
for rank in [1, 2, 3]:
    rank_values = next_task_predictions_long.loc[next_task_predictions_long["rule_rank"].eq(rank), ["snapshot_id", "task_code"]].rename(columns={"task_code": f"top{rank}_task"})
    task_aggregate = task_aggregate.merge(rank_values, on="snapshot_id", how="left", validate="one_to_one")

next_task_predictions = base_snapshots[["snapshot_id", "motorcycle_id"]].merge(
    task_aggregate, on="snapshot_id", how="left", validate="one_to_one"
)
next_task_predictions["prediction_available"] = next_task_predictions["predicted_task_count"].fillna(0).gt(0)
next_task_predictions["prediction_reason"] = np.where(next_task_predictions["prediction_available"], "available", "no due policy task")
next_task_predictions["predicted_task_count"] = next_task_predictions["predicted_task_count"].fillna(0).astype(int)
next_task_predictions["predicted_task_codes"] = next_task_predictions["predicted_task_codes"].fillna("")
next_task_predictions["dataset_version"] = dataset_version
next_task_predictions["rule_version"] = RULE_VERSION
next_task_predictions = next_task_predictions[[
    "snapshot_id", "motorcycle_id", "predicted_task_codes", "top1_task", "top2_task", "top3_task",
    "predicted_task_count", "prediction_available", "prediction_reason", "dataset_version", "rule_version",
]]

timing_output_path = OUTPUTS_DIR / "v0_rule_baseline_next_service_predictions.parquet"
task_output_path = OUTPUTS_DIR / "v0_rule_baseline_next_task_predictions.parquet"
task_long_output_path = OUTPUTS_DIR / "v0_rule_baseline_next_task_predictions_long.parquet"
next_service_predictions.to_parquet(timing_output_path, index=False, engine="pyarrow")
next_task_predictions.to_parquet(task_output_path, index=False, engine="pyarrow")
next_task_predictions_long.to_parquet(task_long_output_path, index=False, engine="pyarrow")

print("Task prediction coverage:", f"{next_task_predictions['prediction_available'].mean():.2%}")
print("Saved:", timing_output_path)
print("Saved:", task_output_path)
print("Saved:", task_long_output_path)


Task prediction coverage: 100.00%
Saved: /Users/nihatkutukoglu/Downloads/ridebase_synthetic_dataset_v1/ridebase-ml/outputs/v0_rule_baseline_next_service_predictions.parquet
Saved: /Users/nihatkutukoglu/Downloads/ridebase_synthetic_dataset_v1/ridebase-ml/outputs/v0_rule_baseline_next_task_predictions.parquet
Saved: /Users/nihatkutukoglu/Downloads/ridebase_synthetic_dataset_v1/ridebase-ml/outputs/v0_rule_baseline_next_task_predictions_long.parquet


## 6. Target alignment, split ve censoring

Target tabloları yukarıdaki prediction fonksiyonuna hiç verilmedi. Yalnız bu evaluation aşamasında
`snapshot_id` ile one-to-one join edilir. Primary split için `split_manifest.csv` authoritative'dir.
Right-censored veya split cutoff ötesindeki satırlar klasik MAE/MedAE hesabına alınmaz.


In [6]:
def key_alignment(left_keys, right_keys, key="snapshot_id"):
    left = pd.DataFrame({key: pd.Series(left_keys).drop_duplicates()})
    right = pd.DataFrame({key: pd.Series(right_keys).drop_duplicates()})
    outer = left.merge(right, on=key, how="outer", indicator=True)
    return {
        "left_duplicates": int(pd.Series(left_keys).duplicated().sum()),
        "right_duplicates": int(pd.Series(right_keys).duplicated().sum()),
        "left_only": int(outer["_merge"].eq("left_only").sum()),
        "right_only": int(outer["_merge"].eq("right_only").sum()),
        "matched": int(outer["_merge"].eq("both").sum()),
    }


timing_alignment = key_alignment(next_service_predictions["snapshot_id"], next_service_targets["snapshot_id"])
task_alignment = key_alignment(next_task_predictions["snapshot_id"], next_task_targets["snapshot_id"])
display(pd.DataFrame([{"alignment": "timing", **timing_alignment}, {"alignment": "task", **task_alignment}]))
if any(timing_alignment[key] for key in ["left_duplicates", "right_duplicates", "left_only", "right_only"]):
    raise RuntimeError("Timing target alignment one-to-one değil")
if any(task_alignment[key] for key in ["left_duplicates", "right_duplicates", "left_only", "right_only"]):
    raise RuntimeError("Task target alignment one-to-one değil")

manifest_columns = [
    "snapshot_id", "primary_time_split", "next_service_regression_eligible_primary",
    "task_target_eligible_primary", "boundary_crossing_future_target",
]
timing_eval = next_service_predictions.merge(
    split_manifest[manifest_columns], on="snapshot_id", how="inner", validate="one_to_one"
).merge(
    next_service_targets[[
        "snapshot_id", "target_event_observed", "is_right_censored", "days_to_next_service",
        "km_to_next_service", "target_km_valid", "target_next_service_at", "next_service_odometer_km",
    ]],
    on="snapshot_id", how="inner", validate="one_to_one"
)

censoring_summary = (
    timing_eval.groupby("primary_time_split")
    .agg(
        total_snapshots=("snapshot_id", "size"),
        observed_targets=("target_event_observed", "sum"),
        censored_targets=("is_right_censored", "sum"),
        timing_prediction_coverage_all=("prediction_available", "mean"),
    )
    .reindex(["TRAIN", "VALIDATION", "TEST"])
    .reset_index()
)
censoring_summary["censoring_rate"] = censoring_summary["censored_targets"] / censoring_summary["total_snapshots"]
observed_coverage = timing_eval.loc[timing_eval["target_event_observed"].eq(1)].groupby("primary_time_split")["prediction_available"].mean()
censored_coverage = timing_eval.loc[timing_eval["is_right_censored"].eq(1)].groupby("primary_time_split")["prediction_available"].mean()
censoring_summary["prediction_coverage_on_observed"] = censoring_summary["primary_time_split"].map(observed_coverage)
censoring_summary["prediction_coverage_on_censored"] = censoring_summary["primary_time_split"].map(censored_coverage)
display(censoring_summary)


,alignment,left_duplicates,right_duplicates,left_only,right_only,matched
0,timing,0,0,0,0,41518
1,task,0,0,0,0,41518


,primary_time_split,total_snapshots,observed_targets,censored_targets,timing_prediction_coverage_all,censoring_rate,prediction_coverage_on_observed,prediction_coverage_on_censored
0,TRAIN,27428,25726,1702,1.0,0.062053,1.0,1.0
1,VALIDATION,6399,4728,1671,1.0,0.261135,1.0,1.0
2,TEST,7691,2622,5069,1.0,0.659082,1.0,1.0


## 7. Next-service timing metrics — ACTIONABLE ana skor

Ana değerlendirme `predicted_days_to_next_service` / `predicted_km_to_next_service` ACTIONABLE
değerlerini kullanır. RAW negatif horizonlar overdue analizi için saklanır. Gözlenmemiş/censored targetlar
regression metriğine hiçbir zaman 0 olarak verilmez.


In [7]:
metrics_rows = []


def add_metric(split, problem, metric, value, n, notes=""):
    metrics_rows.append({
        "split": split, "problem": problem, "metric": metric,
        "value": float(value) if pd.notna(value) else np.nan, "n": int(n), "notes": notes,
    })


def timing_metric_block(split_name, frame):
    observed = frame.loc[
        frame["target_event_observed"].eq(1)
        & frame["next_service_regression_eligible_primary"].eq(1)
    ].copy()
    add_metric(split_name, "NEXT_SERVICE", "total_snapshots", len(frame), len(frame))
    add_metric(split_name, "NEXT_SERVICE", "observed_targets", len(observed), len(frame))
    add_metric(split_name, "NEXT_SERVICE", "censored_targets", int(frame["is_right_censored"].sum()), len(frame))
    add_metric(split_name, "NEXT_SERVICE", "censoring_rate", frame["is_right_censored"].mean(), len(frame))
    add_metric(split_name, "NEXT_SERVICE", "timing_prediction_coverage", observed["prediction_available"].mean(), len(observed), "observed eligible targets")

    days = observed.loc[observed["prediction_available"] & observed["days_to_next_service"].notna()].copy()
    days["days_error"] = days["predicted_days_to_next_service"] - days["days_to_next_service"]
    days["days_abs_error"] = days["days_error"].abs()
    for metric, value in {
        "days_mae": days["days_abs_error"].mean(),
        "days_median_ae": days["days_abs_error"].median(),
        "days_mean_error_bias": days["days_error"].mean(),
        "days_p90_absolute_error": days["days_abs_error"].quantile(.90),
        "days_within_30": days["days_abs_error"].le(30).mean(),
        "days_within_60": days["days_abs_error"].le(60).mean(),
        "days_within_90": days["days_abs_error"].le(90).mean(),
    }.items():
        add_metric(split_name, "NEXT_SERVICE_DAYS", metric, value, len(days), "ACTIONABLE; observed only")

    km = observed.loc[
        observed["prediction_available"] & observed["target_km_valid"].eq(1)
        & observed["km_to_next_service"].notna() & observed["predicted_km_to_next_service"].notna()
    ].copy()
    km["km_error"] = km["predicted_km_to_next_service"] - km["km_to_next_service"]
    km["km_abs_error"] = km["km_error"].abs()
    for metric, value in {
        "km_mae": km["km_abs_error"].mean(),
        "km_median_ae": km["km_abs_error"].median(),
        "km_mean_error_bias": km["km_error"].mean(),
        "km_p90_absolute_error": km["km_abs_error"].quantile(.90),
        "km_within_1000": km["km_abs_error"].le(1_000).mean(),
        "km_within_2000": km["km_abs_error"].le(2_000).mean(),
        "km_within_5000": km["km_abs_error"].le(5_000).mean(),
    }.items():
        add_metric(split_name, "NEXT_SERVICE_KM", metric, value, len(km), "ACTIONABLE; observed valid-km only")
    return days, km


timing_detail_by_split = {}
for split_name in ["TRAIN", "VALIDATION", "TEST"]:
    split_frame = timing_eval.loc[timing_eval["primary_time_split"].eq(split_name)].copy()
    timing_detail_by_split[split_name] = timing_metric_block(split_name, split_frame)

test_days, test_km = timing_detail_by_split["TEST"]
display(pd.DataFrame(metrics_rows))


,split,problem,metric,value,n,notes
0,TRAIN,NEXT_SERVICE,total_snapshots,27428.000000,27428,
1,TRAIN,NEXT_SERVICE,observed_targets,20679.000000,27428,
2,TRAIN,NEXT_SERVICE,censored_targets,1702.000000,27428,
3,TRAIN,NEXT_SERVICE,censoring_rate,0.062053,27428,
4,TRAIN,NEXT_SERVICE,timing_prediction_coverage,1.000000,20679,observed eligible targets
5,TRAIN,NEXT_SERVICE_DAYS,days_mae,155.903229,20679,ACTIONABLE; observed only
6,TRAIN,NEXT_SERVICE_DAYS,days_median_ae,109.728472,20679,ACTIONABLE; observed only
7,TRAIN,NEXT_SERVICE_DAYS,days_mean_error_bias,-155.805714,20679,ACTIONABLE; observed only
8,TRAIN,NEXT_SERVICE_DAYS,days_p90_absolute_error,343.112917,20679,ACTIONABLE; observed only
9,TRAIN,NEXT_SERVICE_DAYS,days_within_30,0.090575,20679,ACTIONABLE; observed only


## 8. Next-task multi-label metrics

Censored task label satırları evaluation dışındadır. Precision/recall/F1, P@1/P@3/R@3/HitRate@3,
subset exact match ve task-count ölçümleri yalnız observed + primary-cutoff eligible snapshotlarda
hesaplanır.


In [8]:
task_label_columns = sorted(column for column in next_task_targets.columns if column.startswith("task__"))
task_codes = [column.removeprefix("task__") for column in task_label_columns]
canonical_target_codes = set(maintenance_tasks["task_code"])
if not set(task_codes).issubset(canonical_target_codes):
    raise RuntimeError("Target task kolonlarında canonical taxonomy dışı code var")

pred_matrix = pd.crosstab(next_task_predictions_long["snapshot_id"], next_task_predictions_long["task_code"]).reindex(
    index=next_task_predictions["snapshot_id"], columns=task_codes, fill_value=0
).gt(0).astype(np.int8)
pred_matrix.index.name = "snapshot_id"
actual_matrix = next_task_targets.set_index("snapshot_id")[task_label_columns].copy()
actual_matrix.columns = task_codes

task_eval_base = next_task_predictions.merge(
    split_manifest[["snapshot_id", "primary_time_split", "task_target_eligible_primary"]],
    on="snapshot_id", how="inner", validate="one_to_one"
).merge(
    next_task_targets[["snapshot_id", "target_event_observed", "is_right_censored"]],
    on="snapshot_id", how="inner", validate="one_to_one"
)


def evaluate_task_ids(ids):
    ids = list(ids)
    if not ids:
        return {}, pd.DataFrame()
    y_true = actual_matrix.loc[ids].fillna(0).astype(np.int8).to_numpy()
    y_pred = pred_matrix.loc[ids].astype(np.int8).to_numpy()
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    micro_precision = tp / (tp + fp) if tp + fp else 0.0
    micro_recall = tp / (tp + fn) if tp + fn else 0.0
    micro_f1 = 2 * micro_precision * micro_recall / (micro_precision + micro_recall) if micro_precision + micro_recall else 0.0

    class_tp = ((y_true == 1) & (y_pred == 1)).sum(axis=0)
    class_fp = ((y_true == 0) & (y_pred == 1)).sum(axis=0)
    class_fn = ((y_true == 1) & (y_pred == 0)).sum(axis=0)
    class_den = 2 * class_tp + class_fp + class_fn
    class_f1 = np.divide(2 * class_tp, class_den, out=np.zeros_like(class_tp, dtype=float), where=class_den > 0)
    support = y_true.sum(axis=0)
    macro_f1 = float(class_f1[support > 0].mean()) if (support > 0).any() else np.nan

    rows = []
    code_to_pos = {code: index for index, code in enumerate(task_codes)}
    ranked = next_task_predictions.set_index("snapshot_id").loc[ids]
    for row_index, snapshot_id in enumerate(ids):
        actual_set = {task_codes[j] for j in np.flatnonzero(y_true[row_index])}
        top = [ranked.loc[snapshot_id, column] for column in ["top1_task", "top2_task", "top3_task"]]
        top = [value for value in top if pd.notna(value) and value in code_to_pos]
        top1_hits = int(bool(top and top[0] in actual_set))
        top3_hits = len(set(top[:3]) & actual_set)
        pred_set = {task_codes[j] for j in np.flatnonzero(y_pred[row_index])}
        rows.append({
            "snapshot_id": snapshot_id,
            "precision_at_1": top1_hits,
            "precision_at_3": top3_hits / 3.0,
            "recall_at_3": top3_hits / len(actual_set) if actual_set else np.nan,
            "hit_rate_at_3": float(top3_hits > 0),
            "subset_exact_match": float(pred_set == actual_set),
            "predicted_task_count": len(pred_set),
            "actual_task_count": len(actual_set),
        })
    row_metrics = pd.DataFrame(rows)
    summary = {
        "task_micro_precision": micro_precision,
        "task_micro_recall": micro_recall,
        "task_micro_f1": micro_f1,
        "task_macro_f1": macro_f1,
        "precision_at_1": row_metrics["precision_at_1"].mean(),
        "precision_at_3": row_metrics["precision_at_3"].mean(),
        "recall_at_3": row_metrics["recall_at_3"].mean(),
        "hit_rate_at_3": row_metrics["hit_rate_at_3"].mean(),
        "subset_exact_match": row_metrics["subset_exact_match"].mean(),
        "average_predicted_task_count": row_metrics["predicted_task_count"].mean(),
        "average_actual_task_count": row_metrics["actual_task_count"].mean(),
    }
    return summary, row_metrics


task_row_metrics_by_split = {}
for split_name in ["TRAIN", "VALIDATION", "TEST"]:
    eligible = task_eval_base.loc[
        task_eval_base["primary_time_split"].eq(split_name)
        & task_eval_base["target_event_observed"].eq(1)
        & task_eval_base["task_target_eligible_primary"].eq(1)
    ]
    ids = eligible["snapshot_id"].tolist()
    summary, row_metrics = evaluate_task_ids(ids)
    task_row_metrics_by_split[split_name] = row_metrics
    coverage = eligible["prediction_available"].mean() if len(eligible) else np.nan
    add_metric(split_name, "NEXT_TASK", "task_prediction_coverage", coverage, len(eligible), "observed eligible targets")
    for metric, value in summary.items():
        note = "macro over TEST-present classes; rare classes are volatile" if metric == "task_macro_f1" else "observed eligible targets"
        add_metric(split_name, "NEXT_TASK", metric, value, len(eligible), note)

baseline_metrics = pd.DataFrame(metrics_rows)
test_task_metrics = task_row_metrics_by_split["TEST"]
display(baseline_metrics.loc[baseline_metrics["split"].eq("TEST")])


,split,problem,metric,value,n,notes
38,TEST,NEXT_SERVICE,total_snapshots,7691.000000,7691,
39,TEST,NEXT_SERVICE,observed_targets,2622.000000,7691,
40,TEST,NEXT_SERVICE,censored_targets,5069.000000,7691,
41,TEST,NEXT_SERVICE,censoring_rate,0.659082,7691,
42,TEST,NEXT_SERVICE,timing_prediction_coverage,1.000000,2622,observed eligible targets
43,TEST,NEXT_SERVICE_DAYS,days_mae,67.410541,2622,ACTIONABLE; observed only
44,TEST,NEXT_SERVICE_DAYS,days_median_ae,57.847917,2622,ACTIONABLE; observed only
45,TEST,NEXT_SERVICE_DAYS,days_mean_error_bias,-67.370904,2622,ACTIONABLE; observed only
46,TEST,NEXT_SERVICE_DAYS,days_p90_absolute_error,125.051736,2622,ACTIONABLE; observed only
47,TEST,NEXT_SERVICE_DAYS,days_within_30,0.173150,2622,ACTIONABLE; observed only


## 9. TEST segment metrics, coverage ve error analysis

Minimum segment örnek eşiği 50'dir. Küçük gruplar raporlanmaz. Error analysis gelecekteki target
değerini yalnız hatayı ölçmek için kullanır; yorumlayıcı kolonlar snapshot-safe geçmiş feature'lardır.


In [9]:
MIN_SEGMENT_N = 50
segment_base = timing_eval.merge(
    base_snapshots[[
        "snapshot_id", "usage_type", "riding_intensity", "brand", "model_name", "workshop_id",
        "motorcycle_age_years", "snapshot_odometer_km", "service_sequence", "previous_failure_count",
    ]],
    on=["snapshot_id", "workshop_id", "snapshot_odometer_km"], how="left", validate="one_to_one",
    suffixes=("", "_feature"),
)
segment_base["age_bucket"] = pd.cut(segment_base["motorcycle_age_years"], [-np.inf, 2, 5, 8, np.inf], labels=["0-2", "3-5", "6-8", "9+"])
segment_base["odometer_bucket"] = pd.cut(segment_base["snapshot_odometer_km"], [-np.inf, 10_000, 30_000, 60_000, 100_000, np.inf], labels=["<10k", "10-30k", "30-60k", "60-100k", "100k+"])
segment_base["service_history_bucket"] = pd.cut(segment_base["service_sequence"], [-np.inf, 1, 4, np.inf], labels=["0-1", "2-4", "5+"])
segment_base["previous_service_count"] = segment_base["service_sequence"].clip(upper=5).astype(str).replace("5", "5+")
segment_base["previous_failure_bucket"] = pd.cut(segment_base["previous_failure_count"], [-np.inf, 0, 1, np.inf], labels=["0", "1", "2+"])

test_task_rows_lookup = test_task_metrics.set_index("snapshot_id") if not test_task_metrics.empty else pd.DataFrame()
segment_rows = []
segment_definitions = {
    "usage_type": "usage_type", "riding_intensity": "riding_intensity", "brand": "brand",
    "model": "model_name", "workshop": "workshop_id", "age_bucket": "age_bucket",
    "odometer_bucket": "odometer_bucket", "previous_service_count": "previous_service_count",
    "previous_failure_count": "previous_failure_bucket", "service_history_bucket": "service_history_bucket",
}
test_segment_base = segment_base.loc[segment_base["primary_time_split"].eq("TEST")].copy()
for segment_type, column in segment_definitions.items():
    for segment_value, group in test_segment_base.groupby(column, observed=True, dropna=False):
        if len(group) < MIN_SEGMENT_N:
            continue
        observed = group.loc[
            group["target_event_observed"].eq(1) & group["next_service_regression_eligible_primary"].eq(1)
        ].copy()
        days_valid = observed.loc[observed["prediction_available"] & observed["days_to_next_service"].notna()]
        km_valid = observed.loc[
            observed["prediction_available"] & observed["target_km_valid"].eq(1)
            & observed["km_to_next_service"].notna() & observed["predicted_km_to_next_service"].notna()
        ]
        task_ids = [snapshot_id for snapshot_id in group["snapshot_id"] if snapshot_id in test_task_rows_lookup.index]
        task_summary, _ = evaluate_task_ids(task_ids)
        segment_rows.append({
            "split": "TEST", "segment_type": segment_type, "segment_value": str(segment_value), "n": len(group),
            "days_mae": (days_valid["predicted_days_to_next_service"] - days_valid["days_to_next_service"]).abs().mean(),
            "km_mae": (km_valid["predicted_km_to_next_service"] - km_valid["km_to_next_service"]).abs().mean(),
            "timing_coverage": observed["prediction_available"].mean() if len(observed) else np.nan,
            "task_micro_f1": task_summary.get("task_micro_f1", np.nan),
            "precision_at_3": task_summary.get("precision_at_3", np.nan),
            "notes": f"minimum_n={MIN_SEGMENT_N}; timing_n={len(observed)}; task_n={len(task_ids)}",
        })
segment_metrics = pd.DataFrame(segment_rows)
display(segment_metrics.head(30))

policy_coverage = (
    base_snapshots.groupby(["model_id", "model_name", "category"], as_index=False)
    .agg(motorcycle_count=("motorcycle_id", "nunique"), snapshot_count=("snapshot_id", "size"))
    .merge(model_policy_summary[["model_id", "applicable_policy_count", "canonical_task_count"]], on="model_id", how="left", validate="one_to_one")
)
timing_cov_model = next_service_predictions.groupby("model_id")["prediction_available"].mean()
task_model = base_snapshots[["snapshot_id", "model_id"]].merge(next_task_predictions[["snapshot_id", "prediction_available"]], on="snapshot_id", validate="one_to_one")
task_cov_model = task_model.groupby("model_id")["prediction_available"].mean()
policy_coverage["timing_prediction_coverage"] = policy_coverage["model_id"].map(timing_cov_model)
policy_coverage["task_prediction_coverage"] = policy_coverage["model_id"].map(task_cov_model)
display(policy_coverage)

test_days_error = test_days.sort_values("days_abs_error", ascending=False).copy()
test_km_error = test_km.sort_values("km_abs_error", ascending=False).copy()
error_analysis = pd.concat([
    test_days_error.head(20).assign(error_type="TOP20_DAYS"),
    test_km_error.head(20).assign(error_type="TOP20_KM"),
], ignore_index=True, sort=False)
display(error_analysis[[column for column in [
    "error_type", "snapshot_id", "usage_type", "riding_intensity", "service_sequence",
    "previous_failure_count", "is_overdue", "days_error", "days_abs_error", "km_error", "km_abs_error",
] if column in error_analysis.columns]])


,split,segment_type,segment_value,n,days_mae,km_mae,timing_coverage,task_micro_f1,precision_at_3,notes
0,TEST,usage_type,COMMUTER,3658,67.291219,4489.931677,1.0,0.246458,0.127588,minimum_n=50; timing_n=1288; task_n=1288
1,TEST,usage_type,COURIER,1490,65.529422,4404.519784,1.0,0.249490,0.127098,minimum_n=50; timing_n=556; task_n=556
2,TEST,usage_type,OFFROAD,142,67.970058,4845.205128,1.0,0.214499,0.128205,minimum_n=50; timing_n=39; task_n=39
3,TEST,usage_type,SEASONAL,793,71.766087,4567.562264,1.0,0.238479,0.145912,minimum_n=50; timing_n=265; task_n=265
4,TEST,usage_type,TOURING,417,70.271884,4676.500000,1.0,0.222892,0.137037,minimum_n=50; timing_n=90; task_n=90
5,TEST,usage_type,TRACK,117,84.275316,5397.606061,1.0,0.223502,0.141414,minimum_n=50; timing_n=33; task_n=33
6,TEST,usage_type,WEEKEND,1074,65.158377,4583.558405,1.0,0.239796,0.114910,minimum_n=50; timing_n=351; task_n=351
7,TEST,riding_intensity,HIGH,4178,60.967512,4783.191284,1.0,0.246095,0.125994,minimum_n=50; timing_n=2180; task_n=2180
8,TEST,riding_intensity,LOW,1089,100.479203,1998.310345,1.0,0.247982,0.155172,minimum_n=50; timing_n=58; task_n=58
9,TEST,riding_intensity,MEDIUM,2424,98.993406,3374.708333,1.0,0.228610,0.136285,minimum_n=50; timing_n=384; task_n=384


,model_id,model_name,category,motorcycle_count,snapshot_count,applicable_policy_count,canonical_task_count,timing_prediction_coverage,task_prediction_coverage
0,BAJAJ_DOMINAR250,Dominar D250,TOURING,57,126,29,29,1.0,1.0
1,BAJAJ_NS125,Pulsar NS125,NAKED,154,452,27,27,1.0,1.0
2,BAJAJ_NS200,Pulsar NS200 UG2,NAKED,209,663,33,33,1.0,1.0
3,CFMOTO_250CLX,250CL-X,ROADSTER,33,72,29,29,1.0,1.0
4,CFMOTO_250DUAL,250 DUAL,ADVENTURE,47,136,29,29,1.0,1.0
5,CFMOTO_250NK,250NK,NAKED,126,357,29,29,1.0,1.0
6,CFMOTO_250SR,250SR,SUPERSPORT,63,155,29,29,1.0,1.0
7,CFMOTO_450CLC,450 CL-C,CRUISER,12,23,26,26,1.0,1.0
8,CFMOTO_450NK,450NK,NAKED,50,143,29,29,1.0,1.0
9,CFMOTO_650NK,650NK,NAKED,23,68,29,29,1.0,1.0


,error_type,snapshot_id,usage_type,riding_intensity,service_sequence,previous_failure_count,is_overdue,days_error,days_abs_error,km_error,km_abs_error
0,TOP20_DAYS,SNP_SVC019332,COMMUTER,MEDIUM,2,0,True,-204.846528,204.846528,NaN,NaN
1,TOP20_DAYS,SNP_SVC032948,COMMUTER,HIGH,4,0,True,-203.000000,203.000000,NaN,NaN
2,TOP20_DAYS,SNP_SVC003585,COMMUTER,HIGH,8,0,True,-202.910417,202.910417,NaN,NaN
3,TOP20_DAYS,SNP_SVC015881,COMMUTER,MEDIUM,4,0,True,-202.901389,202.901389,NaN,NaN
4,TOP20_DAYS,SNP_SVC026762,WEEKEND,LOW,2,0,True,-202.067361,202.067361,NaN,NaN
5,TOP20_DAYS,SNP_SVC013165,SEASONAL,MEDIUM,4,0,True,-200.884028,200.884028,NaN,NaN
6,TOP20_DAYS,SNP_SVC027289,COMMUTER,HIGH,5,0,True,-200.043750,200.043750,NaN,NaN
7,TOP20_DAYS,SNP_SVC027464,COMMUTER,HIGH,3,0,True,-200.006944,200.006944,NaN,NaN
8,TOP20_DAYS,SNP_SVC014386,SEASONAL,MEDIUM,7,0,True,-197.137500,197.137500,NaN,NaN
9,TOP20_DAYS,SNP_SVC003205,SEASONAL,HIGH,5,0,True,-196.943750,196.943750,NaN,NaN


## 10. Leakage audit, machine-readable config ve sanity checks


In [10]:
leakage_audit = pd.DataFrame([
    {"feature_or_input": "snapshot_at", "source": "ml_maintenance_snapshots", "available_at_snapshot": True, "used_by_v0": True, "leakage_status": "PASS"},
    {"feature_or_input": "snapshot_odometer_km", "source": "ml_maintenance_snapshots", "available_at_snapshot": True, "used_by_v0": True, "leakage_status": "PASS"},
    {"feature_or_input": "model/category/powertrain/policy_group", "source": "snapshot + motorcycle model master", "available_at_snapshot": True, "used_by_v0": True, "leakage_status": "PASS"},
    {"feature_or_input": "annual_km_baseline", "source": "usage_profiles", "available_at_snapshot": True, "used_by_v0": True, "leakage_status": "PASS"},
    {"feature_or_input": "historical km/day", "source": "snapshot prior-service feature", "available_at_snapshot": True, "used_by_v0": True, "leakage_status": "PASS"},
    {"feature_or_input": "completed task history <= snapshot_at", "source": "service_tasks + services", "available_at_snapshot": True, "used_by_v0": True, "leakage_status": "PASS"},
    {"feature_or_input": "maintenance policy/taxonomy", "source": "maintenance_policies + maintenance_tasks", "available_at_snapshot": True, "used_by_v0": True, "leakage_status": "PASS"},
    {"feature_or_input": "actual next service date/mileage", "source": "ml_next_service_targets", "available_at_snapshot": False, "used_by_v0": False, "leakage_status": "PASS — evaluation only"},
    {"feature_or_input": "future next-service task labels", "source": "ml_next_task_targets", "available_at_snapshot": False, "used_by_v0": False, "leakage_status": "PASS — evaluation only"},
    {"feature_or_input": "future service/task/status/mileage", "source": "future operational rows", "available_at_snapshot": False, "used_by_v0": False, "leakage_status": "PASS — excluded"},
])
display(leakage_audit)

prediction_task_codes = set(next_task_predictions_long["task_code"])
sanity_checks = pd.DataFrame([
    {"check": "prediction snapshot key duplicate = 0", "value": int(next_service_predictions["snapshot_id"].duplicated().sum()), "status": "PASS" if not next_service_predictions["snapshot_id"].duplicated().any() else "FAIL"},
    {"check": "target join one-to-one", "value": timing_alignment, "status": "PASS" if timing_alignment["matched"] == len(next_service_predictions) else "FAIL"},
    {"check": "no future task-history leakage", "value": int(next_service_predictions["future_history_leak_count"].fillna(0).sum()), "status": "PASS" if next_service_predictions["future_history_leak_count"].fillna(0).sum() == 0 else "FAIL"},
    {"check": "censored excluded from classical metrics", "value": "observed + primary eligible masks", "status": "PASS"},
    {"check": "negative remaining explained as overdue", "value": int(next_service_predictions["is_overdue"].fillna(False).sum()), "status": "PASS"},
    {"check": "no impossible negative predicted odometer", "value": int(next_service_predictions["predicted_next_service_odometer_km"].lt(0).sum()), "status": "PASS" if not next_service_predictions["predicted_next_service_odometer_km"].lt(0).any() else "FAIL"},
    {"check": "predicted task codes canonical", "value": len(prediction_task_codes - canonical_target_codes), "status": "PASS" if prediction_task_codes.issubset(canonical_target_codes) else "FAIL"},
    {"check": "prediction outputs readable", "value": "3 parquet", "status": "PASS"},
    {"check": "all applicable TEST metrics finite", "value": int(np.isfinite(baseline_metrics.loc[baseline_metrics["split"].eq("TEST"), "value"]).sum()), "status": "PASS" if np.isfinite(baseline_metrics.loc[baseline_metrics["split"].eq("TEST"), "value"]).all() else "FAIL"},
])
display(sanity_checks)
if sanity_checks["status"].ne("PASS").any():
    raise RuntimeError("V0 sanity/leakage audit failed")

baseline_config = {
    "dataset_version": dataset_version,
    "rule_version": RULE_VERSION,
    "trained_model": False,
    "deterministic": True,
    "policy_source": "ridebase_v1_2/source_tables/maintenance_policies.csv",
    "taxonomy_source": "ridebase_v1_2/source_tables/maintenance_tasks.csv",
    "date_km_logic": "WHICHEVER_FIRST; km converted with snapshot-safe historical km/day then usage-profile annual baseline",
    "month_days": MONTH_DAYS,
    "overdue_logic": {"raw": "negative retained", "main_evaluation": "ACTIONABLE max(remaining, 0)"},
    "task_ranking_logic": "overdue > due by predicted horizon > safety component > precedence > proximity > task_code",
    "thresholds": {
        "near_due_days": NEAR_DUE_DAYS,
        "near_due_km": NEAR_DUE_KM,
        "offroad_ratio": OFFROAD_RATIO_THRESHOLD,
        "minimum_segment_n": MIN_SEGMENT_N,
    },
    "fallback_behavior": {
        "no_task_history": "observation_start_date + initial_mileage_km baseline",
        "no_historical_km_day": "annual_km_baseline / 365.25",
        "no_applicable_deterministic_policy": "prediction_available=False",
        "condition_only": "not assigned a fabricated due date",
    },
    "target_usage": "evaluation only; never passed to prediction engine",
}
config_path = MODELS_DIR / "v0_rule_baseline_config.json"
config_path.write_text(json.dumps(baseline_config, ensure_ascii=False, indent=2), encoding="utf-8")
print("Config:", config_path)


,feature_or_input,source,available_at_snapshot,used_by_v0,leakage_status
0,snapshot_at,ml_maintenance_snapshots,True,True,PASS
1,snapshot_odometer_km,ml_maintenance_snapshots,True,True,PASS
2,model/category/powertrain/policy_group,snapshot + motorcycle model master,True,True,PASS
3,annual_km_baseline,usage_profiles,True,True,PASS
4,historical km/day,snapshot prior-service feature,True,True,PASS
5,completed task history <= snapshot_at,service_tasks + services,True,True,PASS
6,maintenance policy/taxonomy,maintenance_policies + maintenance_tasks,True,True,PASS
7,actual next service date/mileage,ml_next_service_targets,False,False,PASS — evaluation only
8,future next-service task labels,ml_next_task_targets,False,False,PASS — evaluation only
9,future service/task/status/mileage,future operational rows,False,False,PASS — excluded


,check,value,status
0,prediction snapshot key duplicate = 0,0,PASS
1,target join one-to-one,"{'left_duplicates': 0, 'right_duplicates': 0, 'left_only': 0, 'right_only': 0, 'matched': 41518}",PASS
2,no future task-history leakage,0,PASS
3,censored excluded from classical metrics,observed + primary eligible masks,PASS
4,negative remaining explained as overdue,38848,PASS
5,no impossible negative predicted odometer,0,PASS
6,predicted task codes canonical,0,PASS
7,prediction outputs readable,3 parquet,PASS
8,all applicable TEST metrics finite,31,PASS


Config: /Users/nihatkutukoglu/Downloads/ridebase_synthetic_dataset_v1/ridebase-ml/models/v0_rule_baseline_config.json


## 11. Grafikler


In [11]:
def save_figure(fig, name):
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / name, dpi=170, bbox_inches="tight", facecolor="white")
    plt.close(fig)


def scatter_actual_predicted(frame, actual, predicted, title, xlabel, ylabel, name):
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.scatterplot(data=frame, x=actual, y=predicted, alpha=.35, s=24, ax=ax, color="#175CD3")
    lower = min(frame[actual].min(), frame[predicted].min())
    upper = max(frame[actual].quantile(.99), frame[predicted].quantile(.99))
    ax.plot([lower, upper], [lower, upper], linestyle="--", color="#D92D20", linewidth=1.5, label="Perfect prediction")
    ax.set(xlim=(lower, upper), ylim=(lower, upper), title=title, xlabel=xlabel, ylabel=ylabel)
    ax.legend()
    save_figure(fig, name)


scatter_actual_predicted(test_days, "days_to_next_service", "predicted_days_to_next_service", "TEST: Actual vs predicted days", "Actual days", "Predicted days (ACTIONABLE)", "01_actual_vs_predicted_days.png")
fig, ax = plt.subplots(figsize=(10, 6)); sns.histplot(test_days["days_abs_error"].clip(upper=test_days["days_abs_error"].quantile(.99)), bins=45, ax=ax, color="#7F56D9"); ax.set(title="TEST days absolute error (p99 clipped)", xlabel="Absolute error (days)", ylabel="Snapshots"); save_figure(fig, "02_days_absolute_error_distribution.png")
scatter_actual_predicted(test_km, "km_to_next_service", "predicted_km_to_next_service", "TEST: Actual vs predicted km", "Actual km", "Predicted km (ACTIONABLE)", "03_actual_vs_predicted_km.png")
fig, ax = plt.subplots(figsize=(10, 6)); sns.histplot(test_km["km_abs_error"].clip(upper=test_km["km_abs_error"].quantile(.99)), bins=45, ax=ax, color="#12B76A"); ax.set(title="TEST km absolute error (p99 clipped)", xlabel="Absolute error (km)", ylabel="Snapshots"); save_figure(fig, "04_km_absolute_error_distribution.png")
fig, ax = plt.subplots(figsize=(11, 6)); sns.boxplot(data=test_days_error, x="usage_type", y="days_abs_error", showfliers=False, ax=ax, color="#84CAFF"); ax.set(title="TEST days error by usage type", xlabel="Usage type", ylabel="Absolute error (days)"); ax.tick_params(axis="x", rotation=25); save_figure(fig, "05_error_by_usage_type.png")
history_plot = test_days_error.assign(history_bucket=pd.cut(test_days_error["service_sequence"], [-np.inf, 1, 4, np.inf], labels=["0-1", "2-4", "5+"]))
fig, ax = plt.subplots(figsize=(9, 6)); sns.boxplot(data=history_plot, x="history_bucket", y="days_abs_error", showfliers=False, ax=ax, color="#FEC84B"); ax.set(title="TEST days error by service history", xlabel="Services observed at snapshot", ylabel="Absolute error (days)"); save_figure(fig, "06_error_by_service_history_count.png")
coverage_plot = pd.DataFrame({"Prediction": ["Timing", "Next task"], "Coverage": [next_service_predictions["prediction_available"].mean(), next_task_predictions["prediction_available"].mean()]})
fig, ax = plt.subplots(figsize=(8, 5)); sns.barplot(data=coverage_plot, x="Prediction", y="Coverage", ax=ax, hue="Prediction", legend=False, palette=["#175CD3", "#7F56D9"]); ax.set_ylim(0, 1.05); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set(title="V0 prediction coverage", xlabel="", ylabel="Coverage"); save_figure(fig, "07_prediction_coverage.png")
test_task_summary = baseline_metrics.loc[(baseline_metrics["split"] == "TEST") & baseline_metrics["metric"].isin(["task_micro_precision", "task_micro_recall", "task_micro_f1"])].copy()
fig, ax = plt.subplots(figsize=(9, 5)); sns.barplot(data=test_task_summary, x="metric", y="value", ax=ax, color="#7F56D9"); ax.set_ylim(0, 1); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set(title="TEST next-task micro precision / recall / F1", xlabel="Metric", ylabel="Score"); ax.tick_params(axis="x", rotation=20); save_figure(fig, "08_next_task_precision_recall.png")
top3_plot = baseline_metrics.loc[(baseline_metrics["split"] == "TEST") & baseline_metrics["metric"].isin(["precision_at_1", "precision_at_3", "recall_at_3", "hit_rate_at_3"])].copy()
fig, ax = plt.subplots(figsize=(9, 5)); sns.barplot(data=top3_plot, x="metric", y="value", ax=ax, color="#12B76A"); ax.set_ylim(0, 1); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set(title="TEST top-k task performance", xlabel="Metric", ylabel="Score"); ax.tick_params(axis="x", rotation=20); save_figure(fig, "09_top3_task_performance.png")
fig, ax = plt.subplots(figsize=(10, 6)); overdue_values = next_service_predictions.loc[next_service_predictions["is_overdue"].fillna(False), "overdue_days"]; sns.histplot(overdue_values.clip(upper=overdue_values.quantile(.99)) if len(overdue_values) else overdue_values, bins=40, ax=ax, color="#D92D20"); ax.set(title="Overdue distribution (p99 clipped)", xlabel="Overdue days", ylabel="Snapshots"); save_figure(fig, "10_overdue_distribution.png")

figure_files = sorted(FIGURES_DIR.glob("*.png"))
print("Figure files:", len(figure_files))


Figure files: 10


## 12. Output tabloları ve baseline raporu


In [12]:
metrics_path = REPORT_TABLES_DIR / "v0_rule_baseline_metrics.csv"
segment_path = REPORT_TABLES_DIR / "v0_rule_baseline_segment_metrics.csv"
policy_coverage_path = REPORT_TABLES_DIR / "v0_policy_coverage.csv"
leakage_path = REPORT_TABLES_DIR / "v0_rule_baseline_leakage_audit.csv"
error_path = REPORT_TABLES_DIR / "v0_rule_baseline_error_analysis.csv"
censoring_path = REPORT_TABLES_DIR / "v0_rule_baseline_censoring.csv"
sanity_path = REPORT_TABLES_DIR / "v0_rule_baseline_sanity_checks.csv"

baseline_metrics.to_csv(metrics_path, index=False, encoding="utf-8-sig")
segment_metrics.to_csv(segment_path, index=False, encoding="utf-8-sig")
policy_coverage.to_csv(policy_coverage_path, index=False, encoding="utf-8-sig")
leakage_audit.to_csv(leakage_path, index=False, encoding="utf-8-sig")
error_analysis.to_csv(error_path, index=False, encoding="utf-8-sig")
censoring_summary.to_csv(censoring_path, index=False, encoding="utf-8-sig")
sanity_checks.to_csv(sanity_path, index=False, encoding="utf-8-sig")


def metric_value(metric):
    rows = baseline_metrics.loc[(baseline_metrics["split"] == "TEST") & (baseline_metrics["metric"] == metric), "value"]
    return float(rows.iloc[0]) if len(rows) else np.nan


test_total = int((timing_eval["primary_time_split"] == "TEST").sum())
test_observed = int(((timing_eval["primary_time_split"] == "TEST") & timing_eval["target_event_observed"].eq(1)).sum())
test_censored = int(((timing_eval["primary_time_split"] == "TEST") & timing_eval["is_right_censored"].eq(1)).sum())
policy_motorcycle_coverage = base_snapshots.loc[base_snapshots["model_id"].isin(model_policy_summary.loc[model_policy_summary["applicable_policy_count"].gt(0), "model_id"]), "motorcycle_id"].nunique() / base_snapshots["motorcycle_id"].nunique()
unavailable_reasons = next_service_predictions.loc[~next_service_predictions["prediction_available"], "prediction_reason"].value_counts()
most_common_unavailable = unavailable_reasons.index[0] if len(unavailable_reasons) else "NONE — full timing coverage"
days_bias = metric_value("days_mean_error_bias")
km_bias = metric_value("km_mean_error_bias")

report_text = f"""# Executive Summary

RideBase V0 is a deterministic policy baseline on **Synthetic Dataset v1.2**. It is not a trained model and is not production validation. Production feasibility remains **BLOCKED** because no production extract is available.

# Baseline Definition

Model/group maintenance policies are matched to canonical tasks. The earliest calculable km/date/wear-mean due task triggers the next-service estimate. Main metrics use ACTIONABLE overdue handling (`max(horizon, 0)`); RAW negative horizons remain in prediction output.

# Data and Split

- Dataset version: {dataset_version}
- Total snapshots: {len(base_snapshots):,}
- TEST snapshots: {test_total:,}
- Authoritative split: `split_manifest.csv`
- No training/tuning was performed.

# Policy Coverage

- Motorcycle policy coverage: {policy_motorcycle_coverage:.2%}
- Canonical tasks linked by active policy: {active_policies['task_code'].nunique()}
- Timing prediction coverage: {metric_value('timing_prediction_coverage'):.2%}
- Task prediction coverage: {metric_value('task_prediction_coverage'):.2%}
- Most common unavailable reason: {most_common_unavailable}

# Next-Service Timing Results

TEST observed eligible targets only:

- Days MAE: {metric_value('days_mae'):.3f}
- Days Median AE: {metric_value('days_median_ae'):.3f}
- Days bias: {days_bias:.3f} ({'late' if days_bias > 0 else 'early'} prediction direction)
- Within ±30 / ±60 / ±90 days: {metric_value('days_within_30'):.2%} / {metric_value('days_within_60'):.2%} / {metric_value('days_within_90'):.2%}
- Km MAE: {metric_value('km_mae'):.3f}
- Km Median AE: {metric_value('km_median_ae'):.3f}
- Km bias: {km_bias:.3f} ({'late' if km_bias > 0 else 'early'} prediction direction)
- Within ±1,000 / ±2,000 / ±5,000 km: {metric_value('km_within_1000'):.2%} / {metric_value('km_within_2000'):.2%} / {metric_value('km_within_5000'):.2%}

# Next-Task Results

- Micro precision / recall / F1: {metric_value('task_micro_precision'):.2%} / {metric_value('task_micro_recall'):.2%} / {metric_value('task_micro_f1'):.2%}
- Macro F1: {metric_value('task_macro_f1'):.2%}
- Precision@3 / Recall@3 / HitRate@3: {metric_value('precision_at_3'):.2%} / {metric_value('recall_at_3'):.2%} / {metric_value('hit_rate_at_3'):.2%}
- Subset exact match: {metric_value('subset_exact_match'):.2%}

Macro F1 is volatile for rare task classes. V0 intentionally does not predict unpredictable REPAIR/BREAKDOWN fault tasks; future multi-label models should improve recall without sacrificing precision.

# Segment Analysis

TEST segment metrics use minimum n={MIN_SEGMENT_N}. Small segments are excluded and must not support strong conclusions. See `v0_rule_baseline_segment_metrics.csv`.

# Censoring Handling

TEST: {test_observed:,} observed, {test_censored:,} right-censored. Censored rows are never converted to zero days/km and are excluded from MAE/MedAE and task-label evaluation.

# Error Analysis

The largest 20 day and km errors are exported with snapshot-safe usage/history attributes. Overall days bias is {days_bias:.3f}; km bias is {km_bias:.3f}. Overdue, sparse history and high-usage cases are explicitly identifiable in outputs.

# Leakage Audit

PASS. Prediction inputs are snapshot-time features, policies/taxonomy, and completed task history at or before `snapshot_at`. Next-service and next-task targets enter only after prediction for evaluation.

# Limitations

- Results are synthetic v1.2 offline PoC results, not production performance.
- Production feasibility notebook remains BLOCKED.
- V0 is not designed to predict stochastic failure tasks.
- Maintenance policy quality directly controls baseline quality.
- Censored samples are excluded from classical regression metrics.
- First-history fallback uses observation start + initial mileage and cannot recover unknown pre-observation maintenance.
- Future models must use the same authoritative split and compare against V0.

# Comparison Target for Future Models

V1 Statistical / V2 Survival should primarily beat TEST days/km MAE and Median AE plus coverage/calibration; task models should beat TEST micro-F1 and Recall@3 while maintaining Precision@3.

# Final Verdict

V0 is ready as a deterministic, leakage-audited synthetic benchmark. It does not close Issue #579 production validation.
"""
report_path = REPORTS_DIR / "v0_rule_baseline_report.md"
report_path.write_text(report_text, encoding="utf-8")

print("Metrics:", metrics_path)
print("Segments:", segment_path)
print("Policy coverage:", policy_coverage_path)
print("Leakage audit:", leakage_path)
print("Report:", report_path)


Metrics: /Users/nihatkutukoglu/Downloads/ridebase_synthetic_dataset_v1/ridebase-ml/reports/tables/v0_rule_baseline_metrics.csv
Segments: /Users/nihatkutukoglu/Downloads/ridebase_synthetic_dataset_v1/ridebase-ml/reports/tables/v0_rule_baseline_segment_metrics.csv
Policy coverage: /Users/nihatkutukoglu/Downloads/ridebase_synthetic_dataset_v1/ridebase-ml/reports/tables/v0_policy_coverage.csv
Leakage audit: /Users/nihatkutukoglu/Downloads/ridebase_synthetic_dataset_v1/ridebase-ml/reports/tables/v0_rule_baseline_leakage_audit.csv
Report: /Users/nihatkutukoglu/Downloads/ridebase_synthetic_dataset_v1/ridebase-ml/reports/v0_rule_baseline_report.md


# Issue #579 — V0 Baseline Evidence

Production doğrulaması olmadığı için Issue #579 tamamlandı sayılmaz.


In [13]:
issue_579_evidence = pd.DataFrame([
    {"evidence": "Synthetic baseline implemented", "status": "YES"},
    {"evidence": "Next-service timing benchmark available", "status": "YES"},
    {"evidence": "Next-task rule benchmark available", "status": "YES"},
    {"evidence": "Censoring handled", "status": "YES"},
    {"evidence": "Leakage audit passed", "status": "YES"},
    {"evidence": "Production validation", "status": "BLOCKED"},
    {"evidence": "Ready to compare against V1/V2/V3", "status": "YES"},
])
display(issue_579_evidence)

required_generated_files = [
    timing_output_path, task_output_path, task_long_output_path, config_path, metrics_path, segment_path,
    policy_coverage_path, leakage_path, error_path, censoring_path, sanity_path, report_path, *figure_files,
]
missing_generated = [str(path) for path in required_generated_files if not path.exists()]
if missing_generated:
    raise RuntimeError(f"Generated outputs missing: {missing_generated}")

print("V0_NOTEBOOK_STATUS=PASS")
print("DATASET_VERSION=", dataset_version)
print("TOTAL_SNAPSHOTS=", len(base_snapshots))
print("TEST_SNAPSHOTS=", test_total)
print("TEST_OBSERVED=", test_observed)
print("TEST_CENSORED=", test_censored)
print("Notebook completed with deterministic rules, censoring-safe evaluation and no target leakage.")


,evidence,status
0,Synthetic baseline implemented,YES
1,Next-service timing benchmark available,YES
2,Next-task rule benchmark available,YES
3,Censoring handled,YES
4,Leakage audit passed,YES
5,Production validation,BLOCKED
6,Ready to compare against V1/V2/V3,YES


V0_NOTEBOOK_STATUS=PASS
DATASET_VERSION= 1.2.0
TOTAL_SNAPSHOTS= 41518
TEST_SNAPSHOTS= 7691
TEST_OBSERVED= 2622
TEST_CENSORED= 5069
Notebook completed with deterministic rules, censoring-safe evaluation and no target leakage.


# En Basit Haliyle Bu Notebook Ne Yaptı?

Bu çalışma, motosikletlerin geçmiş servislerini ve bakım kurallarını kullanarak iki soruya basit cevaplar üretmeye çalıştı:

1. **Motosikletin bir sonraki servisi yaklaşık ne zaman gelmeli?**
2. **O serviste hangi bakım işleri yapılabilir?**

Her motosiklet için yalnızca o güne kadar bilinen bilgiler kullanıldı. Gelecekte gerçekten ne olduğu, tahmin hazırlanırken görülmedi. Tahmin yapıldıktan sonra gerçek sonuçlarla karşılaştırıldı.

## Ortaya Çıkan Basit Sonuç

- Kontrol için ayrılan **7.691** kayıt vardı.
- Bunların **2.622** tanesinde bir sonraki servisin ne zaman gerçekleştiği biliniyordu.
- **5.069** tanesinde henüz sonraki servis görülmediği için bunlar hata hesabına katılmadı.
- Sistem bütün kayıtlar için bir servis zamanı ve bakım işi önerisi üretebildi.
- Tahmin edilen servis zamanı, gerçek servisten ortalama **67 gün** farklı çıktı.
- Tahmin edilen servis kilometresi, gerçek değerden ortalama **4.515 km** farklı çıktı.
- Her 100 tahminin yaklaşık **17 tanesi** gerçek servis tarihine 30 gün içinde yaklaştı.
- Her 100 tahminin yaklaşık **52 tanesi** gerçek servis tarihine 60 gün içinde yaklaştı.
- Önerilen ilk üç bakım işi içinde gerçek işlerden en az biri, her 100 servisin yaklaşık **30 tanesinde** bulundu.

## Bu Sonuç Ne Anlama Geliyor?

Bu yöntem son ürün olacak kadar başarılı değildir. Özellikle kilometre ve yapılacak bakım işleri konusunda geliştirilmesi gerekir. Ancak basit, sabit ve herkesin anlayabileceği kurallarla çalışan bir başlangıç noktası oluşturmuştur. Bundan sonra yapılacak daha gelişmiş çalışmaların bu sonuçlardan daha iyi olması beklenir.

## Önemli Not

Bu sonuçlar gerçek müşteri veya gerçek servis verisinden değil, hazırlanmış örnek veriden elde edildi. Bu nedenle gerçek hayattaki başarıyı göstermez. Gerçek şirket verisi henüz olmadığı için gerçek kullanım doğrulaması hâlâ **beklemededir**.
